# Fine-tune on CoNLL-2003 NER

CoNLL-2003 is a good first downstream test because named-entity recognition depends heavily on capitalization. Use this notebook for the capitalization-embedding model, then compare against normal `bert-base-uncased` and `bert-base-cased` runs with matched training settings.

In [ ]:
from pathlib import Path
import os

RUNPOD_REPO = Path("/workspace/repos/CapitalizationEmbeddings")
COLAB_REPO = Path("/content/drive/MyDrive/Github/CapitalizationEmbeddings")
try:
    from google.colab import drive

    if not COLAB_REPO.exists():
        drive.mount("/content/drive")
except Exception:
    pass

if RUNPOD_REPO.exists():
    os.chdir(RUNPOD_REPO)
elif COLAB_REPO.exists():
    os.chdir(COLAB_REPO)

print("repo:", Path.cwd())
%pip install -q -e . -r requirements-colab.txt

from capitalization_embeddings import configure_huggingface_cache
HF_CACHE_DIR = configure_huggingface_cache()
print("HF cache:", HF_CACHE_DIR)


In [ ]:
from pathlib import Path

if not Path("pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the CapitalizationEmbeddings repo root.")

try:
    from google.colab import drive

    drive.mount("/content/drive")
except Exception:
    pass

In [ ]:
# Empty string means initialize from bert-base-uncased plus fresh capitalization embeddings.
# After notebook 01 finishes, set this to checkpoint_dir("mlm") + "/final"
CONTINUED_PRETRAINED_CHECKPOINT = ""

BASE_MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 192
from capitalization_embeddings import checkpoint_dir

OUTPUT_DIR = checkpoint_dir("conll2003_ner")

NUM_EPOCHS = 3
PER_DEVICE_BATCH_SIZE = 16
LEARNING_RATE = 3e-5

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

from capitalization_embeddings import tokenize_with_capitalization

raw = load_dataset("lhoestq/conll2003")
ner_feature = raw["train"].features["ner_tags"].feature
if hasattr(ner_feature, "names"):
    label_list = ner_feature.names
else:
    label_list = [
        "O",
        "B-PER",
        "I-PER",
        "B-ORG",
        "I-ORG",
        "B-LOC",
        "I-LOC",
        "B-MISC",
        "I-MISC",
    ]
id2label = {index: label for index, label in enumerate(label_list)}
label2id = {label: index for index, label in id2label.items()}

tokenizer_name = CONTINUED_PRETRAINED_CHECKPOINT or BASE_MODEL_NAME
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, use_fast=True)

def tokenize_and_align_labels(examples):
    tokenized = tokenize_with_capitalization(
        tokenizer,
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    aligned_labels = []
    for batch_index, word_labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=batch_index)
        previous_word_id = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word_id:
                label_ids.append(word_labels[word_id])
            else:
                label_ids.append(-100)
            previous_word_id = word_id

        aligned_labels.append(label_ids)

    tokenized["labels"] = aligned_labels
    return tokenized

tokenized = raw.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw["train"].column_names,
)

print(label_list)
print(tokenized)

In [ ]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_predictions = [
        [label_list[pred] for pred, label in zip(prediction, label_row) if label != -100]
        for prediction, label_row in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[label] for pred, label in zip(prediction, label_row) if label != -100]
        for prediction, label_row in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
import torch

from capitalization_embeddings import (
    CapitalizedBertConfig,
    CapitalizedBertForTokenClassification,
    DataCollatorForCapitalizedTokenClassification,
    make_trainer,
    make_training_arguments,
)

config_kwargs = {
    "num_labels": len(label_list),
    "id2label": id2label,
    "label2id": label2id,
}

if CONTINUED_PRETRAINED_CHECKPOINT:
    config = CapitalizedBertConfig.from_pretrained(
        CONTINUED_PRETRAINED_CHECKPOINT,
        **config_kwargs,
    )
    model = CapitalizedBertForTokenClassification.from_pretrained(
        CONTINUED_PRETRAINED_CHECKPOINT,
        config=config,
        ignore_mismatched_sizes=True,
    )
else:
    model = CapitalizedBertForTokenClassification.from_uncased_pretrained(
        BASE_MODEL_NAME,
        config_kwargs=config_kwargs,
        ignore_mismatched_sizes=True,
    )

data_collator = DataCollatorForCapitalizedTokenClassification(tokenizer=tokenizer)

training_args = make_training_arguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = make_trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()
print(trainer.evaluate(tokenized["test"]))
trainer.save_model(f"{OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")